In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Resolve repo root whether launched from repo root or Notebooks/
repo_root = Path.cwd().resolve()
if not (repo_root / "backtesting" / "test_001_nvda").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import common

# ── Strategy configuration ───────────────────────────────────────────────
STRATEGY_DIR = repo_root / "backtesting" / "test_001_nvda"
config = common.load_strategy_config(STRATEGY_DIR / "strategy_config.json")

SYMBOL = config["stock_symbol"]
BENCHMARK = "SPY"
MODEL_NAME = config["model_name"]
START = config["sample_start"]
END = config["sample_end"]
PROVIDER = config["data_src"]

# Full indicator spec — all available indicators computed upfront.
INDICATOR_SPEC = {
    "rsi":         {"period": 14},
    "macd":        {"fast": 12, "slow": 26, "signal": 9},
    "cci":         {"period": 20},
    "bbp":         {"period": 20},
    "stochastic":  {"window": 14, "smooth": 3},
    "adx":         {"period": 14},
    "atr":         {"period": 14},
    "obv":         {"signal_period": 20},
    "williams_r":  {"period": 14},
    "ema":         {"period": 50},
    "momentum":    {"period": 10},
}

# Shared feature columns — same 7 indicators used by ALL models for fair comparison.
# Values are relative to SPY (ratio for positive indicators, diff for zero-crossing).
FEATURE_COLUMNS = ["rsi", "macd_hist", "cci", "bbp", "adx", "williams_r", "obv_slope"]

# Which relative transform to use per feature:
# - Ratio (stock/SPY) for indicators that are always positive
# - Difference (stock - SPY) for indicators that cross zero
RATIO_FEATURES = {"rsi", "adx"}
DIFF_FEATURES  = {"macd_hist", "cci", "bbp", "williams_r", "obv_slope"}

print(f"Strategy:  {MODEL_NAME}")
print(f"Symbol:    {SYMBOL} (relative to {BENCHMARK})")
print(f"Window:    {START} -> {END}")
print(f"Features:  {FEATURE_COLUMNS}")
print(f"  ratio:   {sorted(RATIO_FEATURES)}")
print(f"  diff:    {sorted(DIFF_FEATURES)}")

Strategy:  test-001-nvda
Symbol:    NVDA (relative to SPY)
Window:    2021-01-01 -> 2024-12-31
Features:  ['rsi', 'macd_hist', 'cci', 'bbp', 'adx', 'williams_r', 'obv_slope']
  ratio:   ['adx', 'rsi']
  diff:    ['bbp', 'cci', 'macd_hist', 'obv_slope', 'williams_r']


In [ ]:
# ── Fetch OHLCV + compute indicators for stock AND SPY ───────────────────
df_stock = common.fetch_ohlcv(SYMBOL,    start=START, end=END, provider=PROVIDER)
df_spy   = common.fetch_ohlcv(BENCHMARK, start=START, end=END, provider=PROVIDER)

df_stock = common.compute_indicators(df_stock, INDICATOR_SPEC)
df_spy   = common.compute_indicators(df_spy,   INDICATOR_SPEC)

# Inner-join on date to align trading days, then compute relative features.
common_idx = df_stock.index.intersection(df_spy.index)
df_stock = df_stock.loc[common_idx]
df_spy   = df_spy.loc[common_idx]

df = pd.DataFrame(index=common_idx)
df["close"] = df_stock["close"]                          # raw price for equity sim
df["close_spy"] = df_spy["close"]                        # SPY price for benchmark chart
df["rel_price"] = df_stock["close"] / df_spy["close"]    # relative price strength

# ── Relative indicator features (for ML models) ─────────────────────────
for col in FEATURE_COLUMNS:
    if col in RATIO_FEATURES:
        df[col] = df_stock[col] / df_spy[col].replace(0, np.nan)
    else:
        df[col] = df_stock[col] - df_spy[col]

# ── Trend-following features (for Manual Dual Momentum strategy) ─────────
# These encode regime and momentum, where simple thresholds actually work.
df["trend"] = df_stock["close"] / df_stock["close"].ewm(span=50, adjust=False).mean()
df["trend_long"] = df_stock["close"] / df_stock["close"].ewm(span=200, adjust=False).mean()
df["rel_mom_20d"] = df["rel_price"].pct_change(20)       # 20-day relative momentum
df["rel_mom_60d"] = df["rel_price"].pct_change(60)       # 60-day relative momentum

df = df.dropna()

print(f"Bars: {len(df)}  ({df.index[0].date()} -> {df.index[-1].date()})")
print(f"\nTrend features sample:")
df[["close", "rel_price", "trend", "trend_long", "rel_mom_20d", "rel_mom_60d"]].tail()

In [3]:
# ── Inspect relative feature distributions to guide Manual thresholds ────
# Use percentiles to set meaningful buy/sell thresholds.
# Buy signals should fire in the bottom tail; sell in the upper tail.
percentiles = [5, 10, 20, 25, 50, 75, 80, 90, 95]
dist = df[FEATURE_COLUMNS].describe(percentiles=[p/100 for p in percentiles]).T
dist = dist[["mean", "std", "5%", "10%", "20%", "50%", "80%", "90%", "95%"]]
print("Relative feature distributions (stock vs SPY):\n")
dist.round(3)

Relative feature distributions (stock vs SPY):



,mean,std,5%,10%,20%,50%,80%,90%,95%
rsi,1.023,0.172,0.775,0.811,0.880,1.009,1.154,1.236,1.339
macd_hist,0.003,1.258,-2.026,-1.562,-1.054,-0.008,1.006,1.620,2.212
cci,-3.537,85.770,-138.803,-103.745,-67.389,-7.101,55.826,103.041,148.958
bbp,-0.019,0.499,-0.823,-0.606,-0.395,-0.033,0.327,0.601,0.842
adx,1.278,0.582,0.573,0.678,0.803,1.134,1.728,2.048,2.347
williams_r,-4.589,24.746,-49.244,-36.613,-21.592,-2.709,10.982,23.608,36.194
obv_slope,-0.007,2.287,-0.347,-0.161,-0.075,0.031,0.109,0.186,0.357


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# MODEL 1: Manual — Dual Momentum + Trend Following (no ML)
# ══════════════════════════════════════════════════════════════════════════
# Based on Gary Antonacci's Dual Momentum and trend-following principles:
#
#   1. ABSOLUTE momentum: Is the stock trending up? (price > moving average)
#   2. RELATIVE momentum: Is the stock outperforming SPY? (rel_price rising)
#   3. CONFIRMATION: Is momentum accelerating? (MACD, volume)
#
# Buy when trend + relative momentum + at least one confirmation = 3+
# Sell when trend breaks OR relative momentum turns negative = -2+
#
# This works because thresholds on TREND and MOMENTUM features are
# meaningful (unlike thresholds on oscillators which just whipsaw).
MANUAL_SCORE_RULES = [
    # Absolute trend: price above 50-day EMA (buy_above 1.0 = uptrend)
    {"col": "trend",       "buy_above": 1.0,    "sell_below": 0.97},

    # Absolute trend (long): price above 200-day EMA (structural uptrend)
    {"col": "trend_long",  "buy_above": 1.0,    "sell_below": 0.97},

    # Relative momentum (20-day): stock outperforming SPY recently
    {"col": "rel_mom_20d", "buy_above": 0.0,    "sell_below": -0.03},

    # Relative momentum (60-day): stock outperforming SPY over longer window
    {"col": "rel_mom_60d", "buy_above": 0.0,    "sell_below": -0.05},

    # Momentum confirmation: MACD hist diff > 0 = stock momentum leads SPY
    {"col": "macd_hist",   "buy_above": 0,       "sell_below": 0},

    # Volume confirmation: OBV slope diff > 0 = accumulation exceeds market
    {"col": "obv_slope",   "buy_above": 0,       "sell_below": 0},
]

manual_model = common.create_model(
    "manual",
    score_rules=MANUAL_SCORE_RULES,
    buy_threshold=4,    # need trend + relative mom + confirmations (4 of 6)
    sell_threshold=-3,  # trend break + relative mom loss = exit (3 of 6)
)
manual_meta = manual_model.train(df)

manual_signals = manual_model.predict_bulk(df)
print("Manual model — Dual Momentum + Trend Following")
print(f"Signal distribution:")
print(manual_signals.value_counts())
print(f"\n{len(MANUAL_SCORE_RULES)} rules, buy >= 4, sell <= -3")

In [5]:
# ══════════════════════════════════════════════════════════════════════════
# MODEL 2: Classification — BagLearner(RTLearner) bagged decision trees
# ══════════════════════════════════════════════════════════════════════════
# Labels each day by its N-day forward return, then trains a bagged random
# forest to classify buy (+1) / sell (-1) / hold (0).
#
# Strategy-specific params for NVDA:
#   - 10-day lookahead, 4% threshold for label generation
#   - buy_threshold=0.1 / sell_threshold=-0.1 because the ensemble average
#     of {-1, 0, +1} predictions is typically close to 0 in trending markets.
#     The default ±0.5 is too strict and suppresses sell signals entirely.
CLASSIFICATION_PARAMS = {
    "feature_columns": FEATURE_COLUMNS,
    "lookahead": 10,
    "threshold": 0.04,
    "leaf_size": 25,
    "bags": 15,
    "buy_threshold": 0.1,
    "sell_threshold": -0.1,
}

clf_model = common.create_model("classification", **CLASSIFICATION_PARAMS)
clf_meta = clf_model.train(df)

clf_signals = clf_model.predict_bulk(df)
print("Classification model training metadata:")
print(clf_meta)
print(f"\nSignal distribution:")
print(clf_signals.value_counts())

Classification model training metadata:
{'model_type': 'classification', 'train_rows': 978, 'label_distribution': {'long': 437, 'hold': 278, 'short': 263}}

Signal distribution:
buy     679
sell    152
hold    148
Name: count, dtype: int64


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# MODEL 3: Q-Learning — tabular RL with discretized indicator states
# ══════════════════════════════════════════════════════════════════════════
# Two key fixes for Q-Learning with relative indicators:
#
# 1. REWARD: Use rel_price (stock/SPY) as "close" so the reward is
#    relative return. With raw stock returns in a bull market, Q-learner
#    learns "buy and never sell" because returns are always positive.
#    Relative returns can go negative (stock underperforms SPY), giving
#    the Q-learner a real reason to exit.
#
# 2. FEATURES: Use trend-following features that capture regime, not
#    oscillators. Q-Learning with small bins needs features with clear
#    monotonic meaning across bins.
#      - trend: price/50EMA ratio (>1 = uptrend, <1 = downtrend)
#      - rel_mom_20d: 20-day relative momentum (positive = outperforming)
#      - macd_hist: relative MACD histogram (momentum direction)
#
#    3 features x 5 bins = 375 states — dense coverage.

QL_FEATURE_COLUMNS = ["trend", "rel_mom_20d", "macd_hist"]

# Build a Q-Learning–specific df with rel_price as "close" for relative reward
df_ql = df[["rel_price", *QL_FEATURE_COLUMNS]].copy()
df_ql = df_ql.rename(columns={"rel_price": "close"})

QLEARNING_PARAMS = {
    "feature_columns": QL_FEATURE_COLUMNS,
    "n_bins": 5,
    "n_epochs": 500,
    "alpha": 0.15,
    "gamma": 0.95,
    "rar": 0.99,
    "radr": 0.9995,
    "dyna": 0,
}

ql_model = common.create_model("qlearning", **QLEARNING_PARAMS)
ql_meta = ql_model.train(df_ql)

# Predict on the ql dataframe, then map signals back to the main df index
ql_signals = ql_model.predict_bulk(df_ql)
n_states = 5 ** len(QL_FEATURE_COLUMNS) * 3
print(f"Q-Learning: {len(QL_FEATURE_COLUMNS)} features, 5 bins → {n_states} states")
print(f"Reward signal: relative price (stock/SPY) returns")
print(f"Training metadata: {ql_meta}")
print(f"\nSignal distribution:")
print(ql_signals.value_counts())

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Compare all three models
# ══════════════════════════════════════════════════════════════════════════

# ── Strategy-specific simulation parameters ──────────────────────────────
INITIAL_CASH = 10_000          # starting capital
WASH_SALE_DAYS = 30            # cannot buy within N days after a sell
MIN_HOLD_DAYS = 20             # minimum holding period per trade (all models)
MAX_HOLD_DAYS = 150            # force sell after N days (all models)


def apply_constraints(signals, min_hold_days=0, wash_sale_days=0, max_hold_days=0):
    """Filter raw signals to enforce holding period, max hold, and wash-sale rules."""
    out = signals.values.copy()
    dates = signals.index
    position = 0
    entry_idx = None
    last_sell_idx = None

    for i in range(len(out)):
        sig = out[i]

        # Force sell if held too long
        if position == 1 and max_hold_days > 0 and entry_idx is not None:
            if (dates[i] - dates[entry_idx]).days >= max_hold_days:
                out[i] = "sell"
                position = 0
                last_sell_idx = i
                entry_idx = None
                continue

        if sig == "buy" and position == 0:
            if last_sell_idx is not None and (dates[i] - dates[last_sell_idx]).days < wash_sale_days:
                out[i] = "hold"
                continue
            position = 1
            entry_idx = i
        elif sig == "sell" and position == 1:
            if entry_idx is not None and (dates[i] - dates[entry_idx]).days < min_hold_days:
                out[i] = "hold"
                continue
            position = 0
            last_sell_idx = i
            entry_idx = None
        else:
            if (sig == "sell" and position == 0) or (sig == "buy" and position == 1):
                out[i] = "hold"
    return pd.Series(out, index=signals.index)


def simulate_equity(signals, prices, initial_cash=INITIAL_CASH):
    """Simulate a long-only equity curve. Returns (equity, trades)."""
    close = prices.values
    sigs = signals.values
    cash = float(initial_cash)
    shares = 0
    equity = np.empty(len(sigs))
    trades = []
    entry_idx = None

    for i in range(len(sigs)):
        p = close[i]
        if sigs[i] == "buy" and shares == 0:
            shares = int(cash // p)
            cash -= shares * p
            entry_idx = i
        elif sigs[i] == "sell" and shares > 0:
            cash += shares * p
            shares = 0
            if entry_idx is not None:
                trades.append((signals.index[entry_idx], signals.index[i]))
                entry_idx = None
        equity[i] = cash + shares * p
    return pd.Series(equity, index=signals.index), trades


# Apply constraints: all models get wash sale 30d + min hold 20d + max hold 150d
constrained = {
    "Manual": apply_constraints(manual_signals, min_hold_days=MIN_HOLD_DAYS,
                                wash_sale_days=WASH_SALE_DAYS, max_hold_days=MAX_HOLD_DAYS),
    "Classification": apply_constraints(clf_signals, min_hold_days=MIN_HOLD_DAYS,
                                        wash_sale_days=WASH_SALE_DAYS, max_hold_days=MAX_HOLD_DAYS),
    "Q-Learning": apply_constraints(ql_signals, min_hold_days=MIN_HOLD_DAYS,
                                    wash_sale_days=WASH_SALE_DAYS, max_hold_days=MAX_HOLD_DAYS),
}

# ── Charts ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(len(constrained), 1, figsize=(14, 4 * len(constrained)), sharex=True)

for ax, (name, signals) in zip(axes, constrained.items()):
    eq, trades = simulate_equity(signals, df["close"])
    norm_eq = eq / eq.iloc[0]
    benchmark_stock = df["close"] / df["close"].iloc[0]
    benchmark_spy = df["close_spy"] / df["close_spy"].iloc[0]

    ax.plot(norm_eq.index, norm_eq, label=name, linewidth=1.5)
    ax.plot(benchmark_stock.index, benchmark_stock, label=f"{SYMBOL} (Buy & Hold)",
            linestyle="--", color="gray", linewidth=1.5)
    ax.plot(benchmark_spy.index, benchmark_spy, label=f"{BENCHMARK} (Buy & Hold)",
            linestyle="--", color="orange", linewidth=1.5)

    buy_dates = [t[0] for t in trades]
    sell_dates = [t[1] for t in trades]
    if buy_dates:
        ax.scatter(buy_dates, norm_eq.loc[buy_dates], marker="^", color="green",
                   s=80, zorder=5, label="Buy")
    if sell_dates:
        ax.scatter(sell_dates, norm_eq.loc[sell_dates], marker="v", color="red",
                   s=80, zorder=5, label="Sell")

    if trades:
        hold_days = [(s - b).days for b, s in trades]
        avg_hold = np.mean(hold_days)
        std_hold = np.std(hold_days, ddof=1) if len(hold_days) > 1 else 0
        ax.set_title(f"{name}  —  {len(trades)} round-trips, "
                     f"avg hold {avg_hold:.0f}d (std {std_hold:.0f}d)")
    else:
        ax.set_title(f"{name}  —  0 round-trips")

    ax.set_ylabel("Normalized Value")
    ax.legend(loc="upper left", fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── Summary table ────────────────────────────────────────────────────────
summary = []
for name, signals in constrained.items():
    eq, trades = simulate_equity(signals, df["close"])
    total_ret = eq.iloc[-1] / eq.iloc[0] - 1
    daily_rets = eq.pct_change().dropna()
    sharpe = daily_rets.mean() / daily_rets.std() * np.sqrt(252) if daily_rets.std() > 0 else 0
    n_trades = len(trades)
    hold_days = [(s - b).days for b, s in trades]
    avg_hold = f"{np.mean(hold_days):.0f}d" if hold_days else "—"
    std_hold = f"{np.std(hold_days, ddof=1):.0f}d" if len(hold_days) > 1 else "—"
    summary.append({
        "Model": name, "Return": f"{total_ret:.2%}", "Sharpe": f"{sharpe:.2f}",
        "Round-trips": n_trades, "Avg Hold": avg_hold, "Std Hold": std_hold,
    })

bh_ret = df["close"].iloc[-1] / df["close"].iloc[0] - 1
bh_daily = df["close"].pct_change().dropna()
bh_sharpe = bh_daily.mean() / bh_daily.std() * np.sqrt(252)
bh_days = (df.index[-1] - df.index[0]).days
summary.append({
    "Model": f"{SYMBOL} Buy & Hold", "Return": f"{bh_ret:.2%}", "Sharpe": f"{bh_sharpe:.2f}",
    "Round-trips": 1, "Avg Hold": f"{bh_days}d", "Std Hold": "—",
})

spy_ret = df["close_spy"].iloc[-1] / df["close_spy"].iloc[0] - 1
spy_daily = df["close_spy"].pct_change().dropna()
spy_sharpe = spy_daily.mean() / spy_daily.std() * np.sqrt(252)
summary.append({
    "Model": f"{BENCHMARK} Buy & Hold", "Return": f"{spy_ret:.2%}", "Sharpe": f"{spy_sharpe:.2f}",
    "Round-trips": 1, "Avg Hold": f"{bh_days}d", "Std Hold": "—",
})

pd.DataFrame(summary)

In [8]:
# ══════════════════════════════════════════════════════════════════════════
# Export best model
# ══════════════════════════════════════════════════════════════════════════
# Automatically selects the model with the highest Sharpe ratio after
# constraints, cleans up old artifacts, and exports the winner.

import glob

# Pick best model by Sharpe
best_name, best_sharpe, best_model = None, -np.inf, None
model_map = {"Manual": manual_model, "Classification": clf_model, "Q-Learning": ql_model}

for name, signals in constrained.items():
    eq, _ = simulate_equity(signals, df["close"])
    daily_rets = eq.pct_change().dropna()
    sharpe = daily_rets.mean() / daily_rets.std() * np.sqrt(252) if daily_rets.std() > 0 else 0
    if sharpe > best_sharpe:
        best_name, best_sharpe, best_model = name, sharpe, model_map[name]

# Clean up old model artifacts before exporting
old_artifacts = glob.glob(str(STRATEGY_DIR / f"{MODEL_NAME}-*.json"))
for path in old_artifacts:
    Path(path).unlink()
    print(f"  removed {Path(path).name}")

# Export the winner
metadata = best_model.save(STRATEGY_DIR, MODEL_NAME)

print(f"\nBest model: {best_name} (Sharpe {best_sharpe:.2f})")
print(f"Exported to {STRATEGY_DIR}/")
print(f"Artifacts: {list(metadata.get('artifacts', {}).keys())}")

  removed test-001-nvda-policy-metadata.json
  removed test-001-nvda-rf-trees.json

Best model: Classification (Sharpe 2.32)
Exported to /Users/renhao/git/github/RenQuant/backtesting/test_001_nvda/
Artifacts: ['trees']
